# 05. Agentic RAG -- Built directly with LangGraph

We implement Retrieval-Augmented Generation (RAG) in three ways: LangChain RAG Agent, LangChain RAG Chain, and a custom RAG based on LangGraph StateGraph. Covers in-depth patterns such as document relevance evaluation, query rewriting, and conditional routing.

## Learning Objectives

- Understand the overall structure of the RAG pipeline (Indexing -> Search -> Generation)
- Chunk the document with `RecursiveCharacterTextSplitter`
- Build a vector store with `InMemoryVectorStore`
- Implement RAG Agent with LangChain `create_agent` + `@tool`
- Implement RAG Chain (single LLM call) with `@dynamic_prompt` middleware
- Build a custom RAG agent with LangGraph `StateGraph`
- `GradeDocuments` Evaluate document relevance with structured output
- Implement query rewriting and conditional routing

## 5.1 Environment Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-4.1")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("Environment ready.")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically enabled when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 5.2 RAG Overview

Retrieval-Augmented Generation (RAG) is a pattern that improves the accuracy of LLM responses by retrieving external knowledge. LLM has two key limitations:
- **Finite context**: the entire corpus cannot be processed at once.
- **Static Knowledge**: Training data becomes outdated over time.

RAG overcomes this limitation by bringing in relevant external information at query time.

### Pipeline: Indexing -> Search -> Generate

```
[문서] -> Text Splitter -> [청크] -> Embeddings -> [벡터 스토어]
                                                      |
[질문] -> Embedding -> similarity_search -> [관련 청크] -> LLM -> [답변]
```

### 5 Core Components

| Components | Role |
|---|---|
| **Document Loaders** | Collect data from external sources (Google Drive, Notion, etc.) into a standard Document object |
| **Text Splitters** | Split large documents into chunks that fit into the context window |
| **Embedding Models** | Convert text into vectors where semantically similar content is grouped close together |
| **Vector Stores** | A specialized database that stores embeddings and performs similarity searches |
| **Retrievers** | Return related documents based on unstructured queries |

### Three RAG architectures

| Approach | Architecture | LLM Call | Flexibility | Suitable for |
|---|---|---|---|---|
| **2-Step RAG** ​​| Created immediately after search | single | low | FAQ, Doc Bot (fast and predictable) |
| **Agentic RAG** ​​| Agent decides when/how to search | Multiple | High | Complex research, multiple tool approaches |
| **Hybrid RAG** ​​| Query strengthening + search verification + answer quality check | Multiple | High | When repeated purification is required |

### Agent vs Chain approach (LangChain implementation)

| Approach | Architecture | LLM Call | Suitable for |
|---|---|---|---|
| **RAG Agent** | agent + retriever tool | Multiple | Complex queries, query reorganization required |
| **RAG Chain** | Middleware Injection Context | single | Simple Q&A, Predictable Costs |
| **LangGraph Custom** | StateGraph + Custom Node | Multiple | Fine-grained control such as relevance evaluation and rewriting |

## 5.3 Document loading & chunking

### Document Loaders
The document loader reads raw content from various sources and returns it as a `Document` object with fields `page_content` and `metadata`.

| loader | Source | package |
|---|---|---|
| `PyPDFLoader` | PDF file | `pypdf` |
| `TextLoader` | text file | built |
| `CSVLoader` | CSV file | built |
| `WebBaseLoader` | web page | `beautifulsoup4` |
| `DirectoryLoader` | Files in directory | built |

### Text Splitting
`RecursiveCharacterTextSplitter` maintains semantic relevance by recursively splitting it in the order `\n\n` -> `\n` -> ` ` -> `""`. Recommended as the most general purpose splitter.

| parameters | Description | Recommended value |
|---|---|---|
| `chunk_size` | Chunk maximum number of characters | 500-2000 (small for precise search, large for context preservation) |
| `chunk_overlap` | Number of adjacent chunks shared characters | 10-20% of chunk_size (to avoid loss of boundary information) |

### Other dividers

| divider | Suitable for |
|---|---|
| `MarkdownHeaderTextSplitter` | Markdown document |
| `HTMLHeaderTextSplitter` | HTML document |
| `TokenTextSplitter` | Token Budget Based Split |
| `CodeTextSplitter` | Source code (language recognition) |

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

raw_docs = [
    Document(page_content="LangGraph is a state-based multi-actor with LLM"
        "A framework for building applications.",
        metadata={"source": "langgraph-docs"}),
    Document(page_content="Agents use tool to communicate with external systems."
        "Interact. The ReAct pattern alternates between reasoning and action.",
        metadata={"source": "agent-guide"}),
]
print(f"문서 {len(raw_docs)}개 로드됨.")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
)
splits = text_splitter.split_documents(raw_docs)

for i, doc in enumerate(splits):
    print(f"청크 {i}: {doc.page_content[:60]}...")
print(f"총 청크 수: {len(splits)}")

## 5.4 Building a vector store

Vector Store is a specialized database that indexes embeddings and performs similarity searches. `InMemoryVectorStore` is suitable for development/testing.

### Comparison of major vector stores

| Vector Store | Type | Suitable for |
|---|---|---|
| `InMemoryVectorStore` | In-process | Development, small dataset |
| `Chroma` | Embedded/Client-Server | Prototyping, medium-scale datasets |
| `FAISS` | In-process | High performance local search |
| `Pinecone` | Managed Cloud | Production, Scalability |
| `PGVector` | PostgreSQL extensions | Leverage existing PostgreSQL infrastructure |

### Search type

| Search type | Description |
|---|---|
| `"similarity"` | Standard nearest neighbor search |
| `"mmr"` | Maximal Marginal Relevance -- Balancing relevance and diversity (reducing duplication) |
| `"similarity_score_threshold"` | Return only documents with minimum similarity score or higher |

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(
    documents=splits, embedding=embeddings,
)
test_results = vector_store.similarity_search("LangGraph", k=2)
for doc in test_results:
    print(f"  [{doc.metadata['source']}] {doc.page_content[:80]}")
print(f"벡터 스토어 준비 완료. 문서 {len(splits)}개.")

## 5.5 Search tool Definition

Using `response_format="content_and_artifact"` splits the tool output into two parts:
- **Content**: String expression passed to the model (used for inference)
- **Artifact**: The original Document object (accessible programmatically, but not sent to the model)

This separation allows you to use readable text for the model and the original object with metadata for subsequent processing.

In [ ]:
from langchain_core.tools import tool

@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """Search our knowledge base for related articles."""
    docs = vector_store.similarity_search(query, k=4)
    serialized = "\n\n".join(
        f"출처: {d.metadata.get('source', '?')}\n{d.page_content}"
        for d in docs
    )
    return serialized, docs

## 5.6 LangChain RAG Agent -- `create_agent` + `@tool`

Simplest way: register the retriever as tool and call the agent when needed.

### Multi-step search flow
RAG Agent can automatically run multiple discovery steps:
1. **Initial Search** -- Create a query based on user questions
2. **Result evaluation** -- Determine whether the retrieved documents are sufficient for the question
3. **Reorganize and re-search** -- If there are not enough results, modify the query and re-search
4. **Consolidation** -- Combine all search results to create final answer

This approach is suitable for complex research questions, but multiple LLM calls increase costs and delays.

In [ ]:
from langchain.agents import create_agent

rag_agent = create_agent(
    model=llm, tools=[retrieve],
    system_prompt="You are a research assistant."
    "Use retrieve to search before answering.",
)
response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "What is LangGraph?"}]},
    config=lf_config,
)
print(response["messages"][-1].content)

## 5.7 LangChain RAG Chain -- `@dynamic_prompt` Middleware

Implement RAG with a single LLM call. `@dynamic_prompt` retrieves the document before the LLM call and automatically injects it into the system prompt. Because it is a middleware method, it operates in a **single pass** without an agent loop.

| Characteristics | RAG Agent | RAG Chain |
|---|---|---|
| Number of LLM calls | Multiple (agent decision) | single |
| Number of searches | More than once (agent control) | Exactly once (middleware control) |
| Query Reconstruction | automatic | Not supported |
| delay | High (multiple round trips) | Low (single pass) |
| cost | High (more tokens) | Low (fewer tokens) |
| Transparency | Agent inference exposes message | Context injection is implicit |

**Advanced usage**: You can also combine both approaches by injecting the default context with `@dynamic_prompt` while simultaneously providing a retriever tool.

In [ ]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def rag_prompt(request):
    """It retrieves the document and injects it into the system prompt."""
    user_msg = request.state["messages"][-1].content
    docs = vector_store.similarity_search(user_msg, k=4)
    ctx = "\n\n".join(d.page_content for d in docs)
    return f"컨텍스트를 기반으로 답변하세요:\n\n{ctx}"

In [ ]:
chain = create_agent(model=llm, middleware=[rag_prompt])
resp = chain.invoke(
    {"messages": [{"role": "user", "content": "How do agents work?"}]},
    config=lf_config,
)
print(resp["messages"][-1].content)

## 5.8 LangGraph Custom RAG -- Building StateGraph

Build your own RAG agent that allows detailed control with LangGraph `StateGraph`. The key advantage of this approach is that conditional routing allows fine-grained flow control, such as evaluating the relevance of search results and rewriting queries if they are irrelevant.

### Architecture

It is better to limit the maximum number of retries by adding ```
        [generate_query_or_respond]
             /              \
       (tool call)       (no tool call)
           |                  |
      [retrieve]           [END]
           |
   [grade_documents]
      /          \
(relevant)    (not relevant)
    |              |
[generate]   [rewrite_question]
    |              |
  [END]    [generate_query_or_respond]
```

### 각 노드의 역할

| 노드 | 역할 |
|---|---|
| `generate_query_or_respond` | 진입 노드. 검색할지 직접 응답할지 결정 |
| `retrieve` | `ToolNode`로 검색 실행 |
| `grade_documents` | 구조화 출력(`GradeDocuments`)으로 문서 관련성 평가 |
| `rewrite_question` | 관련 없는 결과 시 더 구체적인 쿼리로 리라이트 |
| `generate_answer` | 관련 문서 기반 최종 답변 생성 |

### 무한 루프 방지
`rewrite_question` -> `generate_query_or_respond` 순환이 발생할 수 있으므로, `retry_count` to State. Recommended.

In [ ]:
from langgraph.graph import MessagesState

class AgentState(MessagesState):
    """Custom RAG agent status."""
    relevance: str  # "relevant" or "not_relevant"

print(f"AgentState 키: {list(AgentState.__annotations__)}")

## 5.9 `generate_query_or_respond` node

This is the entry node. Determines whether LLM will call retrieve tool or respond directly.

In [ ]:
llm_with_tools = llm.bind_tools([retrieve])

def generate_query_or_respond(state: AgentState):
    """Decide whether you want to search or respond directly."""
    system = ("You are a useful assistant."
              "For knowledge base questions, use retrieve tool.")
    msgs = [{"role": "system", "content": system}] + state["messages"]
    response = llm_with_tools.invoke(msgs, config=lf_config)
    return {"messages": [response]}

## 5.10 `grade_documents` Node -- Evaluate relevance with structured output

The `GradeDocuments` schema allows LLM to evaluate document relevance. Receive a structured response as `with_structured_output`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class GradeDocuments(BaseModel):
    """Binary relevance score of the retrieved document."""
    relevance: Literal["relevant", "not_relevant"] = Field(
        description="Whether the document is relevant."
    )
    reasoning: str = Field(description="A brief explanation.")

grader = llm.with_structured_output(GradeDocuments)

In [ ]:
def grade_documents(state: AgentState):
    """
    검색된 문서의 관련성을 평가합니다.
    """

    msgs = state["messages"]

    user_q = next(
        (m.content for m in msgs if m.type == "human"),
        ""
    )

    tool_content = msgs[-1].content

    grade = grader.invoke(
        f"질문: {user_q}\n문서:\n{tool_content}\n"
        f"이 문서들이 관련이 있습니까?"
    )

    return {
        "relevance": grade.relevance,
        "messages": msgs
    }

## 5.11 `rewrite_question` node

When retrieved documents are irrelevant, rewrite the original question to be more specific to improve search quality.

In [ ]:
def rewrite_question(state: AgentState):
    """
    검색 품질 향상을 위해 질문을 리라이트합니다.
    """

    user_q = next(
        (m.content for m in state["messages"] if m.type == "human"),
        ""
    )

    prompt = (
        f"다른 용어를 사용하여 리라이트하세요.\n"
        f"원본: {user_q}\n"
        f"리라이트:"
    )

    response = llm.invoke(prompt, config=lf_config)

    return {
        "messages": [
            {
                "role": "human",
                "content": response.content
            }
        ]
    }

## 5.12 `generate_answer` node

Once relevant documents are identified, the search results and the original question are combined to generate the final answer.

In [ ]:
def generate_answer(state: AgentState):
    """
    검색된 문서를 사용하여 답변을 생성합니다.
    """

    user_q = next(
        (m.content for m in state["messages"] if m.type == "human"),
        ""
    )

    docs = next(
        (m.content for m in state["messages"] if m.type == "tool"),
        ""
    )

    prompt = (
        f"컨텍스트만을 사용하여 답변하세요.\n"
        f"컨텍스트:\n{docs}\n\n"
        f"질문: {user_q}"
    )

    resp = llm.invoke(prompt, config=lf_config)

    return {
        "messages": [
            {
                "role": "assistant",
                "content": resp.content
            }
        ]
    }

## 5.13 Graph assembly & execution

Register all nodes in `StateGraph` and connect them with conditional edges (`tools_condition`, `relevance_router`).

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def relevance_router(state: AgentState):
    if state.get("relevance") == "relevant":
        return "generate_answer"
    return "rewrite_question"

graph = StateGraph(AgentState)
graph.add_node("gen_query", generate_query_or_respond)

In [ ]:
graph.add_node("retrieve", ToolNode([retrieve]))
graph.add_node("grade_documents", grade_documents)
graph.add_node("rewrite_question", rewrite_question)
graph.add_node("generate_answer", generate_answer)

graph.add_edge(START, "gen_query")
graph.add_conditional_edges(
    "gen_query", tools_condition,
    {"tools": "retrieve", "__end__": END},
)

In [ ]:
graph.add_edge("retrieve", "grade_documents")
graph.add_conditional_edges(
    "grade_documents", relevance_router,
    {"generate_answer": "generate_answer",
     "rewrite_question": "rewrite_question"},
)
graph.add_edge("rewrite_question", "gen_query")
graph.add_edge("generate_answer", END)

app = graph.compile()
print("Graph compilation successful.")

In [ ]:
for event in app.stream(
    {"messages": [{"role": "user", "content": "What is LangGraph?"}]},
    config=lf_config,
):
    for node_name, output in event.items():
        print(f"--- {node_name} ---")
        if "messages" in output:
            last = output["messages"][-1]
            txt = last.content if hasattr(last, "content") else str(last)
            print(txt[:200])

## Summary

### Comparison of three RAG approaches

| Characteristics | RAG Agent | RAG Chain | LangGraph Custom |
|---|---|---|---|
| Number of LLM calls | Multiple | single | Multiple |
| Number of searches | agent decision | Exactly 1 time | Custom |
| Query Reconstruction | automatic | Not supported | explicit node |
| Relevance Assessment | implicit | None | `GradeDocuments` |
| control level | low | low | High |
| Implementation Complexity | low | Lowest | High |

### Core LangGraph patterns

| pattern | implementation |
|---|---|
| conditional routing | `add_conditional_edges` + `tools_condition` |
| structured output | `llm.with_structured_output(GradeDocuments)` |
| tool node | `ToolNode([retrieve])` |
| loop control | `rewrite_question` -> `gen_query` cycle |

### Next Steps
→ **[06_sql_agent.ipynb](./06_sql_agent.ipynb)**: Creates a SQL agent.